# NCT-CRC retraining with the same model as BreakHis and BACH

**Why:** the NCT-CRC results in the submission came from a different fusion/attention head (concatenation + unshared CBAM MLPs, mixed float16). BreakHis and BACH used the Inception-DenseNet169 model with the branch-level `ChannelAttention` layer in float32. This notebook re-runs **every NCT-CRC experiment** with that BreakHis/BACH model, so all three datasets share one architecture.

**What changes:** only the model (copied verbatim from `Breast Cancer Compression  Study.ipynb`; 36,749,793 parameters) and precision (float32, as for BreakHis/BACH).
**What stays identical to the original NCT-CRC run:** the 7,799-image subset and its split file, the fixed 1,170-image test set, the `StratifiedKFold(5, seed 42)` folds, the shuffle buffer, class weights, optimizer, callbacks, and epochs.

| Job group | Models | Tests on |
|---|---|---|
| Robustness (Exp 2) | 5 | all 7 CRs |
| CAT (Exp 3) | 5 | all 7 CRs |
| CR1×5 control (R2-5) | 5 | all 7 CRs |
| Adaptation (Exp 1) | 35 | its own CR |

**Runtime on L4:** roughly 12–20 h in total. Each model runs in its own subprocess, so memory is released after every model, and is saved as soon as it finishes. **After a disconnect, re-run all cells; finished models are skipped.** You can split the work across several sessions.

**Nothing existing is touched.** Everything goes to `NCT_CRC_SameArch/` in your Drive.

## 0. Setup

In [ ]:
import os, sys, subprocess, json, glob, time, shutil
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'codecarbon'])
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
print('python', sys.version.split()[0])

In [ ]:
# ============================================================
# CONFIG
# ============================================================
DRIVE = '/content/drive/MyDrive'
LOCAL = '/content/nct_local' if IN_COLAB else '/tmp/nct_local'
CFG = {
    'data_dir':   f'{DRIVE}/NCT_CRC_PLOS_ONE/NCT_CRC_Split',
    'split_file': f'{DRIVE}/NCT_CRC_PLOS_ONE/NCT_CRC_Results_Final/nct_crc_subset_splits.csv',
    'out_dir':    f'{DRIVE}/NCT_CRC_SameArch',
    'local_dir':  LOCAL,
    'cr_levels':  ['CR1_original', 'CR15', 'CR25', 'CR50', 'CR100', 'CR500', 'CR1000'],
    'mixed_levels': ['CR1_original', 'CR15', 'CR25', 'CR50', 'CR100'],
    'adaptation_levels': ['CR1_original', 'CR15', 'CR25', 'CR50', 'CR100', 'CR500', 'CR1000'],
    'seed': 42, 'n_folds': 5, 'image_size': [224, 224], 'batch_size': 32, 'epochs': 30, 'learning_rate': 0.001,
    'carbon_country': 'BGD',     # same grid as the BreakHis/BACH runs
    'n_boot': 10000,
    'groups': ['robustness', 'cat', 'cr1x5', 'adaptation'],   # run order; remove a name to skip it
}
OUT = CFG['out_dir']
for sub in ['', 'models', 'predictions', 'records', 'histories', 'emissions', 'results', 'tables']:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
CFG_PATH = os.path.join(OUT, 'config.json')
json.dump(CFG, open(CFG_PATH, 'w'), indent=1)
WORKER = os.path.join(LOCAL, 'nct_worker.py')
os.makedirs(LOCAL, exist_ok=True)
print('outputs ->', OUT)

In [ ]:
# ---- worker script: trains and evaluates ONE model in its own process ----
WORKER_SRC = '"""Train + evaluate ONE NCT-CRC model with the BreakHis/BACH architecture.\n\nCalled by the notebook as:  python nct_worker.py CONFIG.json \'{"exp": ..., "train_cr": ..., "fold": ...}\'\nEverything except the model is copied from \'NCT-CRC JPEG2000 Compression Experiments.ipynb\'\n(same split file, same StratifiedKFold folds, same fixed test set, same training loop).\nModel + precision are copied from \'Breast Cancer Compression  Study.ipynb\' (float32).\nRuns in its own process so all GPU/CPU memory is released when it exits.\n"""\nimport os, sys, json, time\nos.environ.setdefault(\'TF_CPP_MIN_LOG_LEVEL\', \'2\')\nimport numpy as np\nimport pandas as pd\nimport tensorflow as tf\nfrom tensorflow import keras\nfrom tensorflow.keras import layers, Model\nfrom tensorflow.keras.applications import InceptionV3, DenseNet169\nfrom tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau\nfrom sklearn.model_selection import StratifiedKFold\nfrom sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,\n                             f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix)\n\nCFG = json.load(open(sys.argv[1]))\nJOB = json.loads(sys.argv[2])\nSEED = CFG[\'seed\']\nkeras.mixed_precision.set_global_policy(\'float32\')   # as in the BreakHis/BACH runs\n\n# ======================= model: verbatim from the BreakHis/BACH notebooks =======================\ndata_augmentation = keras.Sequential([\n    layers.RandomRotation(0.15),\n    layers.RandomFlip("horizontal"),\n    layers.RandomFlip("vertical"),\n    layers.RandomZoom(0.15),\n], name=\'augmentation\')\n\nclass ChannelAttention(layers.Layer):\n    def __init__(self, ratio=4, **kwargs):\n        super().__init__(**kwargs); self.ratio = ratio\n    def build(self, input_shape):\n        channel = input_shape[-1]\n        self.shared_dense1 = layers.Dense(channel // self.ratio, activation=\'relu\')\n        self.shared_dense2 = layers.Dense(channel, activation=\'sigmoid\')\n        super().build(input_shape)\n    def call(self, x):\n        avg_pool = tf.reduce_mean(x, axis=1, keepdims=True)\n        max_pool = tf.reduce_max(x, axis=1, keepdims=True)\n        avg_out = self.shared_dense2(self.shared_dense1(avg_pool))\n        max_out = self.shared_dense2(self.shared_dense1(max_pool))\n        return x * (avg_out + max_out)\n    def get_config(self):\n        c = super().get_config(); c.update({\'ratio\': self.ratio}); return c\n\ndef build_inception_densenet_attention(input_shape=(224, 224, 3)):\n    inputs = layers.Input(shape=input_shape)\n    x = data_augmentation(inputs)\n    inception = InceptionV3(weights=\'imagenet\', include_top=False, pooling=\'avg\'); inception.trainable = False\n    x1 = inception(x)\n    densenet169 = DenseNet169(weights=\'imagenet\', include_top=False, pooling=\'avg\'); densenet169.trainable = False\n    x2 = densenet169(x)\n    x1 = layers.Dense(512, activation=\'relu\', name=\'inception_proj\')(x1); x1 = layers.BatchNormalization()(x1)\n    x2 = layers.Dense(512, activation=\'relu\', name=\'densenet_proj\')(x2); x2 = layers.BatchNormalization()(x2)\n    stacked = layers.Concatenate(axis=1)([layers.Reshape((1, 512))(x1), layers.Reshape((1, 512))(x2)])\n    fused = layers.Flatten()(ChannelAttention(ratio=4)(stacked))\n    x = layers.BatchNormalization()(fused)\n    x = layers.Dropout(0.5)(x)\n    x = layers.Dense(256, activation=\'relu\')(x)\n    x = layers.Dropout(0.3)(x)\n    outputs = layers.Dense(1, activation=\'sigmoid\')(x)\n    return Model(inputs, outputs, name=\'Inception_DenseNet169_Attention\')\n\nif JOB.get(\'check\'):\n    m = build_inception_densenet_attention()\n    print(json.dumps({\'params\': int(m.count_params()), \'policy\': m.dtype_policy.name,\n                      \'has_ChannelAttention\': any(isinstance(l, ChannelAttention) for l in m.layers),\n                      \'tf\': tf.__version__, \'gpu\': [d.name for d in tf.config.list_physical_devices(\'GPU\')]}))\n    sys.exit(0)\n\n# ======================= data + training loop: verbatim from the NCT-CRC notebook =======================\ndef load_dataset(filepaths, labels, batch_size=32, shuffle=True):\n    @tf.autograph.experimental.do_not_convert\n    def load_and_preprocess(filepath, label):\n        img = tf.io.read_file(filepath)\n        img = tf.image.decode_png(img, channels=3)\n        img = tf.image.resize(img, CFG[\'image_size\'])\n        img = tf.cast(img, tf.float32) / 255.0\n        return img, label\n    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))\n    if shuffle:\n        ds = ds.shuffle(buffer_size=min(len(filepaths), 10000), seed=SEED)\n    return ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)\n\ndef get_class_weights(labels):\n    counts = np.bincount(labels.astype(int)); total = len(labels)\n    return {i: total / (len(counts) * c) for i, c in enumerate(counts)}\n\ndef metrics_of(y_true, prob):\n    y_pred = (prob >= 0.5).astype(int); y_true = y_true.astype(int)\n    try:\n        auc = roc_auc_score(y_true, prob)\n    except ValueError:\n        auc = 0.5\n    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()\n    return {\'accuracy\': accuracy_score(y_true, y_pred), \'balanced_accuracy\': balanced_accuracy_score(y_true, y_pred),\n            \'precision\': precision_score(y_true, y_pred, zero_division=0), \'recall\': recall_score(y_true, y_pred, zero_division=0),\n            \'specificity\': tn / (tn + fp) if (tn + fp) else 0, \'f1_score\': f1_score(y_true, y_pred, zero_division=0),\n            \'mcc\': matthews_corrcoef(y_true, y_pred), \'auc_roc\': auc, \'tp\': int(tp), \'tn\': int(tn), \'fp\': int(fp), \'fn\': int(fn)}\n\ndef local_paths(df, cr):\n    return [os.path.join(CFG[\'local_dir\'], cr, s, c, f)\n            for s, c, f in zip(df[\'original_split\'], df[\'original_class_folder\'], df[\'filename\'])]\n\nsampled = pd.read_csv(CFG[\'split_file\'])\ntv = sampled[sampled[\'split\'].isin([\'train\', \'val\'])].reset_index(drop=True)\nte = sampled[sampled[\'split\'] == \'test\'].reset_index(drop=True)\n\nexp, k, train_cr = JOB[\'exp\'], int(JOB[\'fold\']), JOB[\'train_cr\']\nif exp in (\'adaptation\', \'robustness\'):\n    paths, labels = np.array(local_paths(tv, train_cr)), tv[\'label\'].values\nelif exp == \'cat\':      # cat_combined: CR levels stacked in CAT_LEVELS order\n    paths = np.concatenate([local_paths(tv, c) for c in CFG[\'mixed_levels\']])\n    labels = np.concatenate([tv[\'label\'].values] * len(CFG[\'mixed_levels\']))\nelif exp == \'cr1x5\':    # R2-5 size-matched control: CR1 stacked the same number of times\n    paths = np.concatenate([local_paths(tv, \'CR1_original\')] * len(CFG[\'mixed_levels\']))\n    labels = np.concatenate([tv[\'label\'].values] * len(CFG[\'mixed_levels\']))\nelse:\n    raise ValueError(exp)\n\nskf = StratifiedKFold(n_splits=CFG[\'n_folds\'], shuffle=True, random_state=SEED)\ntr_idx, vl_idx = list(skf.split(paths, labels))[k - 1]\ntag = JOB[\'tag\']\nprint(f\'[{tag}] train {len(tr_idx):,} | val {len(vl_idx):,} | test {len(te):,}\', flush=True)\n\ntf.keras.utils.set_random_seed(SEED)\nmodel = build_inception_densenet_attention()\nmodel.compile(optimizer=keras.optimizers.Adam(learning_rate=CFG[\'learning_rate\']),\n              loss=\'binary_crossentropy\', metrics=[\'accuracy\'])\ncallbacks = [EarlyStopping(monitor=\'val_loss\', patience=5, restore_best_weights=True),\n             ReduceLROnPlateau(monitor=\'val_loss\', factor=0.5, patience=2, min_lr=1e-7)]\n\ntracker = None\ntry:\n    from codecarbon import OfflineEmissionsTracker\n    tracker = OfflineEmissionsTracker(country_iso_code=CFG.get(\'carbon_country\', \'BGD\'), project_name=tag,\n                                      output_dir=os.path.join(CFG[\'out_dir\'], \'emissions\'), log_level=\'error\')\n    tracker.start()\nexcept Exception as e:\n    print(f\'[{tag}] CodeCarbon not active: {e}\', flush=True)\n\nt0 = time.time()\nhist = model.fit(load_dataset(paths[tr_idx], labels[tr_idx], CFG[\'batch_size\'], True),\n                 validation_data=load_dataset(paths[vl_idx], labels[vl_idx], CFG[\'batch_size\'], False),\n                 epochs=CFG[\'epochs\'], callbacks=callbacks, class_weight=get_class_weights(labels[tr_idx]),\n                 verbose=2)\ntrain_s = time.time() - t0\nemissions_kg = None\nif tracker is not None:\n    try:\n        emissions_kg = tracker.stop()\n    except Exception:\n        pass\n\nmodel.save(os.path.join(CFG[\'out_dir\'], \'models\', f\'{tag}.keras\'))\n\ntest_crs = [train_cr] if exp == \'adaptation\' else CFG[\'cr_levels\']\nrows, mets = [], {}\nfor cr in test_crs:\n    prob = model.predict(load_dataset(np.array(local_paths(te, cr)), te[\'label\'].values, 64, False), verbose=0)\n    prob = prob.astype(\'float32\').flatten()\n    mets[cr] = metrics_of(te[\'label\'].values, prob)\n    rows.append(pd.DataFrame(dict(dataset=\'NCT-CRC\', exp=exp, fold=k, train_cr=train_cr, test_cr=cr,\n                                  filename=te[\'filename\'].values,\n                                  patient_id=te[\'filename\'].str.rsplit(\'.\', n=1).str[0].values,\n                                  magnification=\'N/A\', y_true=te[\'label\'].values, prob=prob)))\n    print(f\'[{tag}] test {cr:13s} acc={mets[cr]["accuracy"]:.4f} bal={mets[cr]["balanced_accuracy"]:.4f} \'\n          f\'auc={mets[cr]["auc_roc"]:.4f}\', flush=True)\n\npd.concat(rows).to_csv(os.path.join(CFG[\'out_dir\'], \'predictions\', f\'{tag}.csv.gz\'), index=False, compression=\'gzip\')\njson.dump({kk: [float(v) for v in vv] for kk, vv in hist.history.items()},\n          open(os.path.join(CFG[\'out_dir\'], \'histories\', f\'{tag}.json\'), \'w\'))\nrecord = dict(tag=tag, exp=exp, fold=k, train_cr=train_cr, n_train=int(len(tr_idx)), n_val=int(len(vl_idx)),\n              epochs_trained=len(hist.history[\'loss\']), train_seconds=round(train_s, 1),\n              emissions_g=None if emissions_kg is None else round(emissions_kg * 1000, 4),\n              params=int(model.count_params()), metrics=mets)\n# the record is written last: its existence marks the job as complete\njson.dump(record, open(os.path.join(CFG[\'out_dir\'], \'records\', f\'{tag}.json\'), \'w\'), indent=1)\nprint(f\'[{tag}] DONE in {train_s/60:.1f} min ({record["epochs_trained"]} epochs)\', flush=True)\n'
open(WORKER, 'w').write(WORKER_SRC)
print('worker written:', WORKER, len(WORKER_SRC), 'chars')

## 1. Copy the 7,799-image subset (all 7 CR levels) to the local disk
Read once from Drive (about 55k files, 15–30 min); all training then reads the fast local copy. Already-copied files are skipped.

In [ ]:
def retry(fn, *a, tries=5, wait=3):
    for i in range(tries):
        try:
            return fn(*a)
        except OSError as e:
            if i == tries - 1:
                raise
            time.sleep(wait)

sampled = pd.read_csv(CFG['split_file'])
print('split file:', len(sampled), sampled['split'].value_counts().to_dict())
pairs = []
for cr in CFG['cr_levels']:
    for s, c, f in zip(sampled.original_split, sampled.original_class_folder, sampled.filename):
        pairs.append((os.path.join(CFG['data_dir'], cr, s, c, f), os.path.join(LOCAL, cr, s, c, f)))
def copy_one(p):
    src, dst = p
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return 0
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    retry(shutil.copyfile, src, dst); return 1
t0 = time.time()
with ThreadPoolExecutor(32) as ex:
    n_new = sum(ex.map(copy_one, pairs))
missing = [d for _, d in pairs if not os.path.exists(d)]
print(f'local copy: {len(pairs):,} files ({n_new:,} new) in {time.time() - t0:.0f}s | missing: {len(missing)}')
assert not missing, 'some files could not be copied — re-run this cell'

## 2. Architecture check
The model built by the worker must have exactly the parameter count of the saved BreakHis/BACH models (36,749,793) and run in float32.

In [ ]:
r = subprocess.run([sys.executable, WORKER, CFG_PATH, json.dumps({'check': True, 'exp': 'x', 'fold': 1, 'train_cr': 'x', 'tag': 'x'})],
                   capture_output=True, text=True)
info = json.loads(r.stdout.strip().splitlines()[-1]) if r.returncode == 0 else None
print(info if info else r.stderr[-2000:])
assert info and info['params'] == 36749793 and info['policy'] == 'float32' and info['has_ChannelAttention'], 'architecture check failed'
print('✅ same model as BreakHis/BACH (36,749,793 params, float32, branch-level ChannelAttention)')

## 3. Train all models
Each job prints its epochs and test accuracies. A job whose record file exists is skipped. If a job fails, the loop stops and prints the error so nothing is silently lost.

In [ ]:
jobs = []
for g in CFG['groups']:
    crs = CFG['adaptation_levels'] if g == 'adaptation' else ['Mixed' if g == 'cat' else ('CR1x5' if g == 'cr1x5' else 'CR1_original')]
    for cr in crs:
        for k in range(1, CFG['n_folds'] + 1):
            train_cr = 'CR1_original' if g == 'robustness' else cr
            jobs.append({'exp': g, 'train_cr': train_cr, 'fold': k, 'tag': f'{g}_{train_cr}_fold{k}'})
done = lambda j: os.path.exists(os.path.join(OUT, 'records', j['tag'] + '.json'))
print(f'{len(jobs)} jobs | already done: {sum(map(done, jobs))}')

T0 = time.time()
for i, j in enumerate(jobs, 1):
    if done(j):
        continue
    print(f'\n===== job {i}/{len(jobs)}: {j["tag"]} =====', flush=True)
    p = subprocess.Popen([sys.executable, WORKER, CFG_PATH, json.dumps(j)], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        if ('step - ' in line and 'val_' not in line) or any(k in line for k in ('WARNING', 'absl', 'UserWarning', 'data_adapter')):
            continue
        print(line, end='', flush=True); tail = (tail + [line])[-40:]
    p.wait()
    if p.returncode != 0 or not done(j):
        print(f'\n❌ job {j["tag"]} failed (exit {p.returncode}). Last output:\n' + ''.join(tail))
        break
    print(f'   elapsed this session: {(time.time() - T0)/3600:.2f} h', flush=True)
print(f'\nfinished: {sum(map(done, jobs))}/{len(jobs)} jobs')

## 4. Results
Rebuilds every NCT-CRC result in the formats used in the paper and the repository:
* `results/experiment{1,2,3}_*.csv` and `results/r25_control.csv`: per-fold metrics, same columns as the original NCT-CRC CSVs.
* `tables/`: accuracy tables (mean ± SD), old-vs-new comparison, full metrics, paired statistics (bootstrap + Holm), volume/diversity decomposition, and sustainability.

In [ ]:
recs = [json.load(open(f)) for f in sorted(glob.glob(os.path.join(OUT, 'records', '*.json')))]
print(len(recs), 'records')
LAB = lambda cr: 'CR1' if cr == 'CR1_original' else cr
NAME = {'adaptation': ('experiment1_adaptation.csv', 'exp1_adaptation'), 'robustness': ('experiment2_robustness.csv', 'exp2_robustness'),
        'cat': ('experiment3_compression_aware.csv', 'exp3_cat'), 'cr1x5': ('r25_control.csv', 'r25_cr1x5_control')}
for exp, (fname, ename) in NAME.items():
    rows = []
    for r in recs:
        if r['exp'] != exp:
            continue
        for cr, m in r['metrics'].items():
            rows.append(dict(m, training_time_sec=r['train_seconds'], epochs_trained=r['epochs_trained'], fold=r['fold'],
                             cr_level=LAB(cr), experiment=ename, emissions_g=r['emissions_g']))
    if rows:
        pd.DataFrame(rows).sort_values(['cr_level', 'fold']).to_csv(os.path.join(OUT, 'results', fname), index=False)
        print(f'{fname}: {len(rows)} rows')

P = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(os.path.join(OUT, 'predictions', '*.csv.gz')))], ignore_index=True)
P['correct'] = ((P.prob >= .5).astype(int) == P.y_true).astype(float)
fa = P.groupby(['exp', 'test_cr', 'fold'])['correct'].mean().reset_index()
order = CFG['cr_levels']
acc = (fa.groupby(['exp', 'test_cr'])['correct'].agg(['mean', 'std', 'count']).mul([100, 100, 1]).reset_index())
acc['test_cr'] = pd.Categorical(acc['test_cr'], order, ordered=True)
acc = acc.sort_values(['exp', 'test_cr'])
acc.round(2).to_csv(os.path.join(OUT, 'tables', 'accuracy_mean_sd.csv'), index=False)
piv = acc.pivot(index='test_cr', columns='exp', values='mean').round(2)
print('\nAccuracy (%), mean over folds:'); print(piv.to_string())

In [ ]:
# ---- old (submitted, different architecture) vs new (shared architecture) ----
OLD = {'adaptation': [97.95, 98.34, 97.66, 97.79, 96.68, 85.81, 74.48],
       'robustness': [98.17, 96.68, 90.58, 70.82, 62.96, 60.85, 60.85],
       'cat':        [98.12, 98.28, 98.08, 98.02, 96.24, 61.45, 60.85],
       'cr1x5':      [98.29, 97.37, 87.73, 66.02, 61.78, 60.85, 60.82]}
rows = []
for exp, vals in OLD.items():
    for cr, v in zip(order, vals):
        new = acc[(acc.exp == exp) & (acc.test_cr == cr)]
        if len(new):
            rows.append(dict(exp=exp, test_cr=LAB(cr), old_submitted=v, new_same_arch=round(new['mean'].iloc[0], 2),
                             new_sd=round(new['std'].iloc[0], 2), diff=round(new['mean'].iloc[0] - v, 2)))
CMP = pd.DataFrame(rows); CMP.to_csv(os.path.join(OUT, 'tables', 'old_vs_new_accuracy.csv'), index=False)
print(CMP.to_string(index=False))

In [ ]:
# ---- full metrics per protocol and test CR (fold means) ----
def mets(y, p):
    from sklearn.metrics import roc_auc_score, matthews_corrcoef
    yh = (p >= .5).astype(int); tp = ((yh == 1) & (y == 1)).sum(); tn = ((yh == 0) & (y == 0)).sum()
    fp = ((yh == 1) & (y == 0)).sum(); fn = ((yh == 0) & (y == 1)).sum()
    sens = tp / (tp + fn) if tp + fn else np.nan; spec = tn / (tn + fp) if tn + fp else np.nan
    return dict(accuracy=100 * (tp + tn) / len(y), balanced_accuracy=100 * np.nanmean([sens, spec]), sensitivity=100 * sens,
                specificity=100 * spec, mcc=matthews_corrcoef(y, yh) if len(np.unique(yh)) > 1 else 0.0,
                auc=roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan, pct_pred_positive=100 * yh.mean())
FM = []
for (exp, cr, k), g in P.groupby(['exp', 'test_cr', 'fold']):
    FM.append(dict(exp=exp, test_cr=cr, fold=k, **mets(g.y_true.values, g.prob.values)))
FM = pd.DataFrame(FM).groupby(['exp', 'test_cr']).mean(numeric_only=True).drop(columns='fold').reset_index()
FM['test_cr'] = pd.Categorical(FM['test_cr'], order, ordered=True); FM = FM.sort_values(['exp', 'test_cr']).round(3)
FM.to_csv(os.path.join(OUT, 'tables', 'full_metrics.csv'), index=False)
print(FM.to_string(index=False))

In [ ]:
# ---- paired statistics on the fixed test set (same design as the Tier A analysis) ----
rng = np.random.default_rng(CFG['seed']); B = CFG['n_boot']
def paired(a, b):
    ka = P[(P.exp == a[0]) & (P.test_cr == a[1])]; kb = P[(P.exp == b[0]) & (P.test_cr == b[1])]
    if ka.empty or kb.empty:
        return None
    ga = ka.groupby('filename').agg(ca=('correct', 'mean'), y=('y_true', 'first'))
    gb = kb.groupby('filename').agg(cb=('correct', 'mean'))
    return ga.join(gb, how='inner')
def boot(j):
    d = (j.ca - j.cb).values; n = len(d); pos = (j.y == 1).values
    delta = d.mean()
    W = rng.multinomial(n, np.full(n, 1 / n), size=B).astype(float)
    db = (W @ d) / n
    bal = lambda ca, cb, w: 0.5 * ((w[:, pos] @ (ca - cb)[pos]) / w[:, pos].sum(1) + (w[:, ~pos] @ (ca - cb)[~pos]) / w[:, ~pos].sum(1))
    dbal = bal(j.ca.values, j.cb.values, W)
    S = rng.choice([-1.0, 1.0], size=(B, n))
    p = (1 + np.sum(np.abs((S @ d) / n) >= abs(delta) - 1e-12)) / (B + 1)
    return dict(delta_acc_pp=100 * delta, ci_low=100 * np.percentile(db, 2.5), ci_high=100 * np.percentile(db, 97.5),
                delta_bal_acc_pp=100 * (0.5 * (d[pos].mean() + d[~pos].mean())),
                bal_ci_low=100 * np.nanpercentile(dbal, 2.5), bal_ci_high=100 * np.nanpercentile(dbal, 97.5), p_perm=p, n_images=n)
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); a = np.empty(m); run = 0
    for i, ix in enumerate(o):
        run = max(run, (m - i) * p[ix]); a[ix] = min(1, run)
    return a

comps = []
for c in order[1:]:
    comps += [('within-model', ('robustness', 'CR1_original'), ('robustness', c)), ('within-model', ('cat', 'CR1_original'), ('cat', c))]
for c in order:
    comps.append(('CAT vs robustness', ('cat', c), ('robustness', c)))
for a_, b_ in [('CR1_original', 'CR100'), ('CR1_original', 'CR500'), ('CR100', 'CR500'), ('CR1_original', 'CR1000'), ('CR500', 'CR1000')]:
    comps.append(('adaptation (between-model)', ('adaptation', a_), ('adaptation', b_)))
rows = []
for fam, a, b in comps:
    j = paired(a, b)
    if j is not None:
        rows.append(dict(family=fam, A=f'{a[0]}:{LAB(a[1])}', B=f'{b[0]}:{LAB(b[1])}', **boot(j)))
ST = pd.DataFrame(rows); ST['p_holm'] = holm(ST.p_perm.values); ST = ST.round(4)
ST.to_csv(os.path.join(OUT, 'tables', 'paired_comparisons.csv'), index=False)
pd.set_option('display.width', 250); print(ST.to_string(index=False))

In [ ]:
# ---- R2-5: volume effect (control − robustness) and diversity effect (CAT − control) ----
rows = []
for c in order:
    for eff, a, b in [('volume', ('cr1x5', c), ('robustness', c)), ('diversity', ('cat', c), ('cr1x5', c))]:
        j = paired(a, b)
        if j is not None:
            rows.append(dict(effect=eff, test_cr=LAB(c), **boot(j)))
VD = pd.DataFrame(rows)
if len(VD):
    VD['p_holm'] = holm(VD.p_perm.values); VD = VD.round(4)
    VD.to_csv(os.path.join(OUT, 'tables', 'r25_volume_vs_diversity.csv'), index=False)
    print(VD[['effect', 'test_cr', 'delta_acc_pp', 'ci_low', 'ci_high', 'p_holm']].to_string(index=False))

# ---- sustainability (CodeCarbon, Bangladesh grid as for BreakHis/BACH) ----
R = pd.DataFrame([{k: r[k] for k in ['tag', 'exp', 'fold', 'train_cr', 'epochs_trained', 'train_seconds', 'emissions_g', 'n_train']} for r in recs])
R.to_csv(os.path.join(OUT, 'tables', 'training_record.csv'), index=False)
SU = R.groupby('exp').agg(models=('tag', 'count'), train_min=('train_seconds', lambda s: s.sum() / 60), co2_g=('emissions_g', 'sum')).round(2)
SU.loc['total'] = SU.sum(); SU.to_csv(os.path.join(OUT, 'tables', 'sustainability.csv'))
print('\n', SU.to_string())
print(f'\nSend me the folder {OUT}/tables/ (and results/). Models and predictions stay in {OUT}.')